# Disclosure Gap Matrix

This notebook generates the publication-ready disclosure gap matrix. All company classifications are loaded from the `data/disclosure_coding.csv` table; no company-level disclosure values are hard-coded here.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from pathlib import Path

WORKBOOK_NAME = "disclosure_coding.csv"

def find_plot_project():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "plots_for_paper"]
    for candidate in candidates:
        if (candidate / "data" / WORKBOOK_NAME).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate data/{WORKBOOK_NAME}")

PROJECT_DIR = find_plot_project()
DATA_FILE = PROJECT_DIR / "data" / WORKBOOK_NAME
PLOTS_DIR = PROJECT_DIR / "plots"
plot_data = pd.read_csv(DATA_FILE).sort_values("Plot order")

# Five index conditions (columns), in the order used in the manuscript.
# Facility-level and Region-resolved are merged into "Spatially attributable electricity".
FIELDS = [
    "DC-only MWh",
    "Spatially attributable electricity",
    "Hourly CFE",
    "Boundary",
    "Renewable accounting transparency",
]
CONDITION_LABELS = [
    "Data-center-only\nelectricity\n(MWh)",
    "Spatially\nattributable\nelectricity",
    "Time-based\nalignment\n(hourly CFE)",
    "Boundary\nclarity",
    "Renewable-\nelectricity\naccounting",
]
COMPANY_LABELS = {"Amazon": "Amazon/AWS"}

companies = plot_data["Company"].tolist()
status = plot_data.set_index("Company").loc[companies, FIELDS]
unknown = set(status.stack().dropna().unique()) - {"Yes", "Partial", "No"}
if unknown or status.isna().any().any():
    raise ValueError(f"Missing or unknown disclosure status values: {sorted(unknown)}")

matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['font.size'] = 8

GREEN, GREEN_EDGE = '#4CAF50', '#388E3C'
BLUE, BLUE_EDGE = '#4FC3F7', '#0288D1'
GREY_EDGE = '#9E9E9E'
MARKER = 13  # points; markers stay circular regardless of axis scaling

def draw_status(ax, x, y, value):
    style = dict(marker='o', markersize=MARKER, markeredgewidth=0.9, linestyle='none', zorder=3)
    if value == "Yes":
        ax.plot(x, y, markerfacecolor=GREEN, markeredgecolor=GREEN_EDGE, **style)
    elif value == "Partial":
        ax.plot(x, y, markerfacecolor=BLUE, markerfacecoloralt='white', fillstyle='left',
                markeredgecolor=BLUE_EDGE, **style)
    else:
        ax.plot(x, y, markerfacecolor='white', markeredgecolor=GREY_EDGE, **style)

n_rows, n_cols = len(companies), len(FIELDS)
COL0, DX = 1.9, 1.15          # x of first condition column and column spacing
x_cols = [COL0 + j * DX for j in range(n_cols)]
x_right = x_cols[-1] + 0.6
fig, ax = plt.subplots(figsize=(7.2, 3.0))
ax.set_xlim(0, x_right)
ax.set_ylim(-1.2, n_rows + 0.9)
ax.axis('off')

# Column headers: condition number above a short condition name.
top = n_rows - 0.5
for x, j, label in zip(x_cols, range(n_cols), CONDITION_LABELS):
    ax.text(x, top + 1.3, f"{j + 1}", ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.text(x, top + 1.25, label, ha='center', va='top', fontsize=7.5, linespacing=1.1)
ax.text(0.05, top + 1.3, "Company", ha='left', va='bottom', fontsize=9, fontweight='bold')
ax.plot([0, x_right], [top, top], color='#757575', lw=0.8)

# Rows: one per company.
for i, company in enumerate(companies):
    y = n_rows - 1 - i
    if i % 2 == 0:
        ax.add_patch(Rectangle((0, y - 0.5), x_right, 1.0, facecolor='#F3F4F6', edgecolor='none', zorder=0))
    ax.text(0.05, y, COMPANY_LABELS.get(company, company), ha='left', va='center', fontsize=8.5)
    for x, field in zip(x_cols, FIELDS):
        draw_status(ax, x, y, status.loc[company, field])
ax.plot([0, x_right], [-0.5, -0.5], color='#757575', lw=0.8)

legend_elements = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor=GREEN, markeredgecolor=GREEN_EDGE, markersize=8, label='Yes'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor=BLUE, markeredgecolor=BLUE_EDGE, fillstyle='left', markerfacecoloralt='white', markersize=8, label='Partial'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor='white', markeredgecolor=GREY_EDGE, markersize=8, label='No'),
]
ax.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, 0.0), ncol=3,
          frameon=False, fontsize=8, handletextpad=0.3, columnspacing=1.6)

fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01)
fig.savefig(PLOTS_DIR / "disclosure_gap_matrix.pdf", format='pdf', dpi=600, facecolor='white', bbox_inches='tight', pad_inches=0.03)
fig.savefig(PLOTS_DIR / "disclosure_gap_matrix.png", format='png', dpi=600, facecolor='white', bbox_inches='tight', pad_inches=0.03)
plt.show()
print(f"Loaded {len(companies)} companies from {DATA_FILE.name} and saved disclosure_gap_matrix.pdf/.png")
